# Between-cycle gait variability (E3.4)

**Purpose:** Evaluate whether between-cycle angle-dispersion descriptors add predictive information.

**Required inputs:** Prepared Baček HDF5; EXT-S0 results ZIP; original reference ZIP; exact approved method and E3.4 annex documents.

**Outputs:** Four model prediction ledgers, paired comparisons and verification records.

**Usage:** Run after EXT-S0. Required source notebook filename: E3_4_Between_Cycle_Variability_Local.ipynb. Files are read from the notebook’s current working directory unless specified in its configuration.


In [ ]:
# 1 — Fixed approval identity, local paths, software, and immutable checksums
from pathlib import Path
from datetime import datetime, timezone
import csv, hashlib, io, json, platform, sys, zipfile
from collections import Counter

import h5py
import numpy as np
import scipy
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

H5 = Path('bacek_clean_common_angles_metabolic_dataset.h5')
EXT_ZIP = Path('EXT_S0_Shared_Preflight_results.zip')
REFERENCE_ZIP = Path('EXT_S0_Original_Reference_Records.zip')
APPROVED_AMENDMENT = Path('Third_FYP_E3_4_E5_3_Joint_Protocol_Amendment_v0.2_APPROVAL_READY.md')
APPROVED_E34 = Path('Third_FYP_E3_4_Between_Cycle_Variability_Proposed_Experimental_Protocol_v0.2.md')
NOTEBOOK = Path('E3_4_Between_Cycle_Variability_Local.ipynb')
OUT = Path('E3_4_Between_Cycle_Variability_results')
ARCHIVE = Path('E3_4_Between_Cycle_Variability_results.zip')

METHOD_ID = 'FYP-JA-2026-11-04-02'
SOURCE_SHA = 'c73ce561e94de52f1dc86128829c43ef536c4cf6b5f4aa9af1ea060409221fa3'
EXT_ZIP_SHA = 'deb90024d9b67766161e651548b51d8a6ef99f104421155152ea9b0357ec2522'
REFERENCE_ZIP_SHA = '5fd8b80d99522f82c3d2f45000dc553edf9afe5b125ea862d7da9bb89dba33ce'
BOOTSTRAP_SHA = '5d93bf4ef62f3d488243f0b1732c8260fde78e8e1e1dba7afe6fb79390660373'
AMENDMENT_SHA = 'd9d0741420392592f30b79c7fe3812d7259b728b33914ad280227b73b5cbd1db'
E34_SHA = 'c45dfe60b2a14754a5c65a71157a5dca86349919cfb2340585c6499f8852e222'
CHANNELS = ('hip_left','hip_right','knee_left','knee_right','ankle_left',
            'ankle_right','pelvis_sagittal','pelvis_frontal','pelvis_transverse')
ALPHAS = (0.01,0.1,1.0,10.0,100.0,1000.0)
PROCEDURES = ('D','D+V','C+D','C+D+V')
ATOL_DEG = 1e-10
PRED_ATOL = 1e-12
METRIC_ATOL = 1e-10


def check(value, message):
    if not bool(value):
        raise AssertionError(message)


def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(8*1024*1024), b''):
            h.update(block)
    return h.hexdigest()


def rows_csv(blob):
    return list(csv.DictReader(io.StringIO(blob.decode('utf-8-sig'))))


def save_csv(path, rows, columns):
    with path.open('w', encoding='utf-8', newline='') as f:
        w = csv.DictWriter(f, fieldnames=columns)
        w.writeheader()
        w.writerows(rows)


def fmt(x):
    return format(float(x), '.17g')


for f in (H5, EXT_ZIP, REFERENCE_ZIP, APPROVED_AMENDMENT, APPROVED_E34, NOTEBOOK):
    check(f.is_file(), f'Missing required file: {f}')
for f, expected in ((H5,SOURCE_SHA),(EXT_ZIP,EXT_ZIP_SHA),
                    (REFERENCE_ZIP,REFERENCE_ZIP_SHA),
                    (APPROVED_AMENDMENT,AMENDMENT_SHA),(APPROVED_E34,E34_SHA)):
    check(sha256_file(f)==expected, f'SHA-256 mismatch: {f} — STOP')

with zipfile.ZipFile(EXT_ZIP) as z:
    names={Path(n).name:n for n in z.namelist() if not n.endswith('/')}
    def ext(name): return z.read(names[name])
    ext_summary=json.loads(ext('EXT_S0_summary.json'))
    check(ext_summary['status']=='PASS' and ext_summary['approved_method_release']==METHOD_ID,
          'Previous EXT-S0 result not accepted as PASS for this approved release')
    check(ext_summary['source_hdf5_sha256']==SOURCE_SHA, 'EXT-S0 source identity mismatch')
    ext_blobs={name:ext(name) for name in ('canonical_trial_ledger.csv',
                'canonical_W_ALL9.csv','canonical_D_ALL9.csv',
                'canonical_original_cycle_indices.csv',
                'outer_fold_manifest.csv','inner_fold_manifest.csv')}
    for name,blob in ext_blobs.items():
        check(hashlib.sha256(blob).hexdigest()==ext_summary['files_sha256'][name],
              f'EXT-S0 exported CSV altered: {name}')

with zipfile.ZipFile(REFERENCE_ZIP) as z:
    ref_inner_blob=z.read('04/inner_fold_manifest.csv')
    ref_outer_blob=z.read('04/outer_fold_manifest.csv')
    foundational=json.loads(z.read('04/run_summary.json'))
    previous_predictions=rows_csv(z.read('04/held_out_predictions.csv'))
    previous_models=rows_csv(z.read('04/selected_models.csv'))
    bootstrap_blob=z.read('14/bootstrap_resampled_participants.csv')
check(hashlib.sha256(bootstrap_blob).hexdigest()==BOOTSTRAP_SHA,
      'The original E5.2 bootstrap schedule is missing or modified')
check(foundational['status']=='PASS' and foundational['source_sha256']==SOURCE_SHA,
      'Original foundational run identity mismatch')
check(ext_blobs['inner_fold_manifest.csv']==ref_inner_blob and
      ext_blobs['outer_fold_manifest.csv']==ref_outer_blob,
      'Original participant partitions changed')

print('Frozen release, corrected source, EXT-S0 and original references: PASS')
print('Software: Python', sys.version.split()[0], '| NumPy',np.__version__,
      '| SciPy',scipy.__version__,'| scikit-learn',sklearn.__version__,'| h5py',h5py.__version__)

## Feature, source and participant checks


In [ ]:
# 2 — Load and strictly validate the accepted canonical trial, W and D ledgers
trial_records=rows_csv(ext_blobs['canonical_trial_ledger.csv'])
W_records=rows_csv(ext_blobs['canonical_W_ALL9.csv'])
D_records=rows_csv(ext_blobs['canonical_D_ALL9.csv'])
outer_records=rows_csv(ref_outer_blob)
inner_records=rows_csv(ref_inner_blob)
index_records=rows_csv(ext_blobs['canonical_original_cycle_indices.csv'])

UIDS=[r['trial_uid'] for r in trial_records]
check(len(UIDS)==284 and len(set(UIDS))==284, 'Expected 284 unique primary trial UIDs')
check([r['trial_uid'] for r in W_records]==UIDS and
      [r['trial_uid'] for r in D_records]==UIDS, 'Canonical W/D trial order mismatch')
people=np.asarray([r['subject_id'] for r in trial_records])
y=np.asarray([float(r['observed_W_per_kg']) for r in trial_records], dtype=np.float64)
fold_ids=np.asarray([int(r['outer_fold']) for r in trial_records], dtype=int)
N=np.asarray([int(r['retained_cycles']) for r in trial_records], dtype=int)
condition_idx=np.asarray([int(r['condition_index']) for r in trial_records], dtype=int)
check(len(set(people))==21 and N.sum()==60757 and N.min()>=100 and N.max()<=348,
      'Primary source population/count mismatch')
check(np.isfinite(y).all() and (y>0).all(), 'Invalid corrected metabolic targets')
check(np.all((0<=condition_idx)&(condition_idx<15)), 'Unexpected condition index')
check(all(int(r['condition_index'])==int(r['trial_id'][1:])-1 for r in trial_records),
      'T01–T15 condition mapping mismatch')
C=np.eye(15,dtype=np.float64)[condition_idx]

Wcols=list(W_records[0].keys())[1:]
Dcols=list(D_records[0].keys())[1:]
check(len(Wcols)==909 and len(Dcols)==54, 'Original 909/54 feature column count mismatch')
check(Wcols==[f'{c}_phase_{p:03d}' for c in CHANNELS for p in range(101)],
      'Canonical waveform channel/phase order mismatch')
check(Dcols==[f'{c}_{desc}' for c in CHANNELS for desc in
                   ('q0','q1','q2','q3','phase_sd','rom')],
      'Canonical descriptor channel/descriptor order mismatch')
Wref=np.asarray([[float(r[c]) for c in Wcols] for r in W_records], dtype=np.float64)
Dref=np.asarray([[float(r[c]) for c in Dcols] for r in D_records], dtype=np.float64)
check(Wref.shape==(284,909) and Dref.shape==(284,54) and
      np.isfinite(Wref).all() and np.isfinite(Dref).all(), 'Invalid frozen W/D matrices')

outer={int(r['outer_fold']): r for r in outer_records}
inner={(int(r['outer_fold']),int(r['inner_fold'])): r for r in inner_records}
check(len(outer)==5 and len(inner)==20, 'Incomplete five outer/twenty inner partitions')
all_people=set(people.tolist())

def listed_people(s):
    return set(x for x in s.split(';') if x)

for f in range(1,6):
    row=outer[f]
    dev=listed_people(row['development_participants'])
    test=listed_people(row['test_participants'])
    check(test.isdisjoint(dev) and dev|test==all_people, f'Outer fold {f} overlap/omission')
    check(np.array_equal(fold_ids==f,np.isin(people,list(test))),
          f'Outer fold {f} trial IDs differ from participant assignment')
    check(sum(fold_ids==f)==int(row['n_test_trials']) and
          sum(fold_ids!=f)==int(row['n_development_trials']),
          f'Outer fold {f} count mismatch')
    covered=set()
    for j in range(1,5):
        r=inner[(f,j)]
        tr=listed_people(r['train_participants'])
        va=listed_people(r['validation_participants'])
        check(tr.isdisjoint(va) and tr|va==dev and
              listed_people(r['outer_test_participants'])==test,
              f'Inner fold {f}/{j} participant contract violation')
        vi=set(np.flatnonzero(np.isin(people,list(va))).tolist())
        check(not covered.intersection(vi), f'Inner fold {f}/{j} repeated validation rows')
        covered.update(vi)
        check(sum(np.isin(people,list(tr)))==int(r['n_train_trials']) and
              len(vi)==int(r['n_validation_trials']),
              f'Inner fold {f}/{j} trial count mismatch')
    check(covered==set(np.flatnonzero(fold_ids!=f).tolist()),
          f'Outer fold {f} incomplete inner validation coverage')

check(len(index_records)==60757, 'Wrong original-index ledger length')
print('Accepted canonical trial order, W/D column order, conditions and all original folds: PASS')

In [ ]:
# 3 — Independent mathematical definitions and constructed-data arithmetic tests

def mean_waveform(arr):
    return np.asarray(arr,dtype=np.float64).mean(axis=0,dtype=np.float64)


def descriptors(m):
    result=[]
    for line in m:
        for q in range(4):
            piece=line[25*q:25*q+26]
            result.append(float((0.5*piece[0]+np.sum(piece[1:-1],dtype=np.float64)
                                 +0.5*piece[-1])/25.0))
        result.extend((float(np.std(line,ddof=0)),float(np.ptp(line))))
    return np.asarray(result,dtype=np.float64)


def dispersion_fast(arr):
    # Sample variance across cycles at each phase; then root of mean phase variance.
    return float(np.sqrt(np.mean(np.var(np.asarray(arr,dtype=np.float64),
                                        axis=0,ddof=1),dtype=np.float64)))


def dispersion_independent(arr):
    # Independent two-pass calculation, loops across the 101 phase indices.
    x=np.asarray(arr,dtype=np.float64)
    n=x.shape[0]
    variances=[]
    for p in range(101):
        phase=x[:,p]
        centre=float(sum(float(v) for v in phase))/n
        ss=sum((float(v)-centre)**2 for v in phase)
        variances.append(ss/(n-1))
    return float(np.sqrt(sum(variances)/101.0))


z=np.zeros((3,101),dtype=np.float64)
unit=np.stack([z[0]-1.0,z[1],z[2]+1.0])
constant=np.ones((4,101),dtype=np.float64)*7.5
varying=np.zeros((3,101),dtype=np.float64)
varying[:,0]=(-2.0,0.0,2.0)
for x,expect in ((unit,1.0),(constant,0.0),(varying,np.sqrt(4.0/101.0))):
    check(abs(dispersion_fast(x)-expect)<=1e-12,'Fast synthetic variability fixture failed')
    check(abs(dispersion_independent(x)-expect)<=1e-12,
          'Independent synthetic variability fixture failed')
check(abs(dispersion_fast(varying)-np.mean(np.std(varying,axis=0,ddof=1)))>0.1,
      'Synthetic fixture fails to distinguish phase root-mean-variance')
check(abs(dispersion_fast(unit[[2,0,1]])-dispersion_fast(unit))<=1e-12,
      'Cycle-order invariance synthetic fixture failed')
print('Synthetic independent V arithmetic, unequal-phase weighting and order checks: PASS')

In [ ]:
# 4 — Compute all V descriptors locally and enforce full-trial EXT-S0 numerical equality
V_rows=[]; local_D=[]; local_W=[]; local_index=[]
uid_to_idx={uid:i for i,uid in enumerate(UIDS)}

# Original-cycle vector from accepted EXT-S0 output, grouped in immutable trial order.
indices_by_uid={uid:[] for uid in UIDS}
for r in index_records:
    check(r['trial_uid'] in indices_by_uid, 'Unexpected UID in original index ledger')
    indices_by_uid[r['trial_uid']].append((int(r['retained_row_0based']),
                                            int(r['original_cycle_number'])))
for uid,n in zip(UIDS,N):
    seq=indices_by_uid[uid]
    check(len(seq)==n and [a for a,b in seq]==list(range(n)),
          f'Original-cycle index row ordering invalid for {uid}')
    check(all(seq[k+1][1]>seq[k][1] for k in range(n-1)),
          f'Original-cycle indices non-increasing for {uid}')

with h5py.File(H5,'r') as h:
    check(sorted(h['subjects'].keys())==[f'S{i:02d}' for i in range(1,22)],
          'Unexpected HDF5 participant population')
    for i,uid in enumerate(UIDS):
        sid,tid=uid.split('_')
        g=h['subjects'][sid]['trials'][tid]
        n=int(N[i]); source_arrays=[]; vv=[]
        raw_index=np.asarray(g['original_cycle_number'][()]).reshape(-1)
        check(raw_index.size==n and np.all(np.isfinite(raw_index)) and
              np.all(raw_index==np.floor(raw_index)),f'{uid}: invalid source indices')
        check(np.array_equal(raw_index.astype(np.int64),
              np.asarray([v for r,v in indices_by_uid[uid]],dtype=np.int64)),
              f'{uid}: original-cycle index mismatch')
        target=np.asarray(g['metcost_mean'][()])
        check(target.size==1 and np.isfinite(target).all() and
              abs(float(target.reshape(-1)[0])-y[i])<=1e-12,
              f'{uid}: corrected stored target mismatch')
        for ch in CHANNELS:
            x=np.asarray(g[ch][()],dtype=np.float64)
            check(x.shape==(n,101) and np.isfinite(x).all(),
                  f'{uid}/{ch}: invalid retained-cycle source')
            source_arrays.append(x)
            v1=dispersion_fast(x)
            v2=dispersion_independent(x)
            check(np.isfinite(v1) and v1>=0 and abs(v1-v2)<=ATOL_DEG,
                  f'{uid}/{ch}: V implementations disagree (>1e-10 degrees)')
            check(abs(v1-dispersion_fast(x[::-1]))<=ATOL_DEG and
                  np.allclose(mean_waveform(x),mean_waveform(x[::-1]),
                              atol=ATOL_DEG,rtol=0),
                  f'{uid}/{ch}: retained-row-order invariance failed')
            vv.append(v1)
        M=np.stack([mean_waveform(x) for x in source_arrays])
        d=descriptors(M)
        reversed_M=np.stack([mean_waveform(x[::-1]) for x in source_arrays])
        check(np.allclose(d,descriptors(reversed_M),atol=ATOL_DEG,rtol=0),
              f'{uid}: D row-order invariance failed')
        check(np.allclose(M.reshape(909),Wref[i],atol=ATOL_DEG,rtol=0),
              f'{uid}: full-trial mean differs from canonical EXT-S0')
        check(np.allclose(d,Dref[i],atol=ATOL_DEG,rtol=0),
              f'{uid}: frozen descriptor differs from canonical EXT-S0')
        V_rows.append(vv)
        local_W.append(M.reshape(909))
        local_D.append(d)

V=np.asarray(V_rows,dtype=np.float64)
check(V.shape==(284,9) and np.isfinite(V).all() and (V>=0).all(),
      'Invalid 284×9 V-ALL9 features')
check(np.allclose(np.vstack(local_W),Wref,atol=ATOL_DEG,rtol=0),
      'Whole-population canonical W equality failed')
check(np.allclose(np.vstack(local_D),Dref,atol=ATOL_DEG,rtol=0),
      'Whole-population canonical D equality failed')
print('EXT-S0 E3.4 source-level cross-equality: PASS — 284×909 W, 284×54 D, 284×9 V.')

In [ ]:
# 5 — Export mandatory pre-outcome V quality-control records before any model fitting
OUT.mkdir(exist_ok=True)
Vcols=[f'{ch}_between_cycle_dispersion_deg' for ch in CHANNELS]
V_manifest=[{'trial_uid':uid,'subject_id':people[i],'retained_cycles':int(N[i]),
             **{Vcols[j]:fmt(V[i,j]) for j in range(9)}}
            for i,uid in enumerate(UIDS)]
save_csv(OUT/'V_ALL9_by_trial.csv',V_manifest,
         ['trial_uid','subject_id','retained_cycles']+Vcols)

qc=[]
for j,ch in enumerate(CHANNELS):
    col=V[:,j]
    span=float(np.max(col)-np.min(col))
    cc=float(np.corrcoef(col,N)[0,1]) if np.std(col)>0 and np.std(N)>0 else None
    qc.append({'channel':ch,'minimum_degrees':fmt(np.min(col)),
               'median_degrees':fmt(np.median(col)),
               'maximum_degrees':fmt(np.max(col)),
               'zero_trial_count':int(np.count_nonzero(col==0)),
               'exact_constant_column':bool(span==0),
               'near_constant_column':bool(span<=1e-10),
               'pearson_V_vs_retained_count':fmt(cc) if cc is not None else 'undefined_constant_vector'})
save_csv(OUT/'V_ALL9_preoutcome_QC.csv',qc,list(qc[0].keys()))
pre_qc={'status':'PASS','gate':'E3.4_PRE_OUTCOME_FEATURE_GATE',
        'method_release':METHOD_ID,'local_ext_s0_cross_feature_equality':'PASS',
        'V_shape':[284,9],'V_arithmetic_independent':'PASS',
        'V_manifest_sha256':sha256_file(OUT/'V_ALL9_by_trial.csv'),
        'V_QC_sha256':sha256_file(OUT/'V_ALL9_preoutcome_QC.csv'),
        'model_scoring_started':False}
(OUT/'E3_4_preoutcome_QC_summary.json').write_text(json.dumps(pre_qc,indent=2)+'\n',encoding='utf-8')
print('Pre-outcome V manifest and channel QC exported. No rows or features excluded.')

## Participant-grouped predictive evaluation


In [ ]:
# 6 — Frozen four feature contracts, metrics, and fitting-only preprocessing
X_blocks={'D':Dref, 'D+V':np.column_stack((Dref,V)),
          'C+D':np.column_stack((C,Dref)),
          'C+D+V':np.column_stack((C,Dref,V))}
check({key:X.shape for key,X in X_blocks.items()}==
      {'D':(284,54),'D+V':(284,63),'C+D':(284,69),'C+D+V':(284,78)},
      'Procedure feature counts mismatch')


def pipeline(procedure, alpha):
    if procedure.startswith('C+'):
        pre=ColumnTransformer([('conditions','passthrough',slice(0,15)),
                               ('angles',StandardScaler(),slice(15,None))],
                              remainder='drop', sparse_threshold=0)
        return Pipeline([('preprocess',pre),('ridge',Ridge(alpha=alpha,fit_intercept=True))])
    return Pipeline([('scale',StandardScaler()),
                     ('ridge',Ridge(alpha=alpha,fit_intercept=True))])


def constant_count(X_train,procedure):
    data=X_train[:,15:] if procedure.startswith('C+') else X_train
    return int(np.count_nonzero(np.ptp(data,axis=0)==0))


def rmse(actual,pred):
    e=np.asarray(pred,dtype=np.float64)-np.asarray(actual,dtype=np.float64)
    return float(np.sqrt(np.mean(e*e,dtype=np.float64)))


def metrics(actual,pred):
    a=np.asarray(actual,dtype=np.float64);p=np.asarray(pred,dtype=np.float64)
    e=p-a
    denom=float(np.sum((a-a.mean())**2,dtype=np.float64))
    check(denom>0,'Invalid R2 denominator')
    return {'n_trials':len(a),'RMSE_W_per_kg':rmse(a,p),
            'MAE_W_per_kg':float(np.mean(np.abs(e),dtype=np.float64)),
            'bias_W_per_kg':float(np.mean(e,dtype=np.float64)),
            'R2':float(1-np.sum(e*e,dtype=np.float64)/denom)}


# Outcome-blind synthetic pipeline exercise: fixed feature shapes and repeat fit.
rng=np.random.default_rng(2026)
for name in PROCEDURES:
    Xmock=rng.normal(size=(14,X_blocks[name].shape[1]))
    if name.startswith('C+'):
        Xmock[:,:15]=np.eye(15)[np.arange(14)]
    ymock=rng.normal(size=14)
    m=pipeline(name,1.0).fit(Xmock[:10],ymock[:10])
    pred1=m.predict(Xmock[10:]);pred2=m.predict(Xmock[10:])
    check(np.isfinite(pred1).all() and np.max(np.abs(pred1-pred2))<=PRED_ATOL,
          f'Synthetic Ridge repeatability failed: {name}')
print('Synthetic Ridge pipeline/prediction repeatability: PASS')

In [ ]:
# 7 — Prespecified pooled inner validation, deterministic alpha selection and held-out prediction
candidate_scores=[]; selected_models=[]; predictions=[]; fit_checks=[]
started_utc=datetime.now(timezone.utc).isoformat()

for f in range(1,6):
    dev_idx=np.flatnonzero(fold_ids!=f)
    test_idx=np.flatnonzero(fold_ids==f)
    trainable=set(dev_idx.tolist())
    splits=[]
    for j in range(1,5):
        ir=inner[(f,j)]
        tr_idx=np.flatnonzero(np.isin(people,list(listed_people(ir['train_participants']))))
        va_idx=np.flatnonzero(np.isin(people,list(listed_people(ir['validation_participants']))))
        check(set(tr_idx).isdisjoint(va_idx) and
              set(tr_idx).union(va_idx)==trainable,
              f'Inner participant overlap/incomplete split: {f}/{j}')
        splits.append((tr_idx,va_idx))
    check(set(np.concatenate([v for t,v in splits]))==trainable and
          len(np.concatenate([v for t,v in splits]))==len(dev_idx),
          f'Inner validation coverage failed for outer fold {f}')

    for name in PROCEDURES:
        X=X_blocks[name]
        pooled=[]
        for alpha in ALPHAS:
            inner_oof=np.full(len(y),np.nan,dtype=np.float64)
            for j,(train_idx,val_idx) in enumerate(splits,start=1):
                mdl=pipeline(name,alpha)
                mdl.fit(X[train_idx],y[train_idx])
                vals=np.asarray(mdl.predict(X[val_idx]),dtype=np.float64)
                check(np.isfinite(vals).all(),f'{name}, f{f}/j{j}: nonfinite inner prediction')
                inner_oof[val_idx]=vals
                fit_checks.append({'stage':'inner','procedure_id':name,'outer_fold':f,
                                   'inner_fold':j,'alpha':fmt(alpha),
                                   'n_fit_trials':len(train_idx),
                                   'constant_continuous_columns':constant_count(X[train_idx],name)})
            check(np.isfinite(inner_oof[dev_idx]).all(),
                  f'Incomplete pooled inner prediction: {name}, outer fold {f}')
            score=rmse(y[dev_idx],inner_oof[dev_idx])
            row={'outer_fold':f,'procedure_id':name,'alpha':fmt(alpha),
                 'inner_pooled_RMSE_W_per_kg':fmt(score)}
            candidate_scores.append(row)
            pooled.append((score,alpha))
        # Exact float64 score tie goes to smaller alpha; candidate order remains frozen.
        chosen_score,chosen_alpha=min(pooled,key=lambda pair:(pair[0],pair[1]))
        selected_models.append({'outer_fold':f,'procedure_id':name,
                                'selected_alpha':fmt(chosen_alpha),
                                'inner_pooled_RMSE_W_per_kg':fmt(chosen_score),
                                'n_continuous_features':X.shape[1]-(15 if name.startswith('C+') else 0),
                                'constant_outer_dev_continuous_columns':constant_count(X[dev_idx],name)})

        fitted=pipeline(name,chosen_alpha).fit(X[dev_idx],y[dev_idx])
        first=np.asarray(fitted.predict(X[test_idx]),dtype=np.float64)
        # N1: independent deterministic fit of the selected complete pipeline, same data/runtime.
        replay=pipeline(name,chosen_alpha).fit(X[dev_idx],y[dev_idx])
        second=np.asarray(replay.predict(X[test_idx]),dtype=np.float64)
        difference=float(np.max(np.abs(first-second)))
        check(np.isfinite(first).all() and np.isfinite(second).all()
              and difference<=PRED_ATOL,
              f'N1 model prediction repeatability failed: {name}, outer {f}, {difference}')
        fit_checks.append({'stage':'outer_repeatability','procedure_id':name,'outer_fold':f,
                           'inner_fold':'','alpha':fmt(chosen_alpha),
                           'n_fit_trials':len(dev_idx),
                           'constant_continuous_columns':constant_count(X[dev_idx],name),
                           'max_pred_difference_W_per_kg':fmt(difference)})
        for loc,idx in enumerate(test_idx):
            predictions.append({'experiment_id':'E3.4','procedure_id':name,
                                'subject_id':str(people[idx]),
                                'trial_id':trial_records[idx]['trial_id'],
                                'trial_uid':UIDS[idx], 'outer_fold':f,
                                'target_W_per_kg':fmt(y[idx]),
                                'prediction_W_per_kg':fmt(first[loc]),
                                'input_contract':name,'selected_alpha':fmt(chosen_alpha)})
    print(f'Outer fold {f}: {len(test_idx)} trial predictions for each of four procedures — PASS')

check(len(candidate_scores)==5*4*6 and len(selected_models)==20 and
      len(predictions)==4*284,'E3.4 score/selection/prediction count mismatch')
print('Grouped Ridge evaluation and four matched prediction ledgers complete.')

In [ ]:
# 8 — Verify complete held-out coverage; strict original D comparison; contextual C-only reference
pred_lookup={(r['procedure_id'],r['trial_uid']):r for r in predictions}
check(len(pred_lookup)==4*284,'Duplicate procedure/UID held-out predictions')
for name in PROCEDURES:
    check({u for p,u in pred_lookup if p==name}==set(UIDS),
          f'Missing held-out trial predictions: {name}')
    for i,uid in enumerate(UIDS):
        r=pred_lookup[(name,uid)]
        check(int(r['outer_fold'])==int(fold_ids[i]) and
              abs(float(r['target_W_per_kg'])-y[i])<=1e-12,
              f'Unmatched target/fold for {name}/{uid}')

old_D={r['trial_uid']:r for r in previous_predictions if r['procedure']=='D-ALL9_FIXED'}
old_C={r['trial_uid']:r for r in previous_predictions if r['procedure']=='B2'}
check(len(old_D)==284 and len(old_C)==284,'Missing original D/B2 prediction references')
old_D_selection={int(r['outer_fold']):r for r in previous_models
                 if r['procedure']=='D-ALL9_FIXED'}
check(len(old_D_selection)==5,'Original D alpha selection archive incomplete')
for record in selected_models:
    if record['procedure_id']=='D':
        f=int(record['outer_fold']); earlier=old_D_selection[f]
        check(float(record['selected_alpha'])==float(earlier['alpha']),
              f'Original D alpha not reproduced at outer fold {f}')
        check(abs(float(record['inner_pooled_RMSE_W_per_kg'])-
                  float(earlier['inner_pooled_RMSE_W_per_kg']))<=PRED_ATOL,
              f'Original D pooled inner score not reproduced at outer fold {f}')

max_diff_D=0.0
for i,uid in enumerate(UIDS):
    p=pred_lookup[('D',uid)]
    prior=old_D[uid];c=old_C[uid]
    check(int(prior['outer_fold'])==fold_ids[i] and int(c['outer_fold'])==fold_ids[i],
          f'Original reference fold mismatch: {uid}')
    check(abs(float(prior['observed_W_per_kg'])-y[i])<=PRED_ATOL and
          abs(float(c['observed_W_per_kg'])-y[i])<=PRED_ATOL,
          f'Original reference target mismatch: {uid}')
    max_diff_D=max(max_diff_D,abs(float(p['prediction_W_per_kg'])-
                                  float(prior['predicted_W_per_kg'])))
check(max_diff_D<=PRED_ATOL,
      f'Original D-ALL9 replay fails strict 1e-12 reference threshold: {max_diff_D}')

C_predictions=np.asarray([float(old_C[uid]['predicted_W_per_kg']) for uid in UIDS])
original_C_rmse=rmse(y,C_predictions)
check(abs(original_C_rmse-0.5587655313821055)<=PRED_ATOL,
      'Original C/B2 reference RMSE unexpected; do not replace original C model')
print('Exactly matched D original procedure: PASS; maximum prediction deviation:',max_diff_D)
print('Original, unchanged condition-only C/B2 contextual RMSE:',original_C_rmse)

In [ ]:
# 9 — Original E5.2 bootstrap, paired contrasts, participant-balanced and per-person summaries
boot_rows=rows_csv(bootstrap_blob)
check(len(boot_rows)==10000,'Original E5.2 draw count mismatch')
participants=sorted(all_people)
check(participants==[f'S{i:02d}' for i in range(1,22)],'Unexpected sampled-participant universe')
p_idx={name:i for i,name in enumerate(participants)}
counts=np.zeros((10000,21),dtype=np.int16)
for i,r in enumerate(boot_rows):
    sampled=r['population_sampled_participants'].split(';')
    check(int(r['draw_index'])==i+1 and len(sampled)==21 and
          set(sampled).issubset(all_people) and
          len(set(sampled))==int(r['population_distinct_participants']),
          f'Incorrect E5.2 fixed participant bootstrap draw {i+1}')
    for p in sampled:
        counts[i,p_idx[p]]+=1
check(np.all(counts.sum(axis=1)==21),'Bootstrap draw multiplicity mismatch')

preds={name:np.asarray([float(pred_lookup[(name,u)]['prediction_W_per_kg']) for u in UIDS],
                       dtype=np.float64) for name in PROCEDURES}
pooled_metrics=[]
for name in PROCEDURES:
    pooled_metrics.append({'procedure_id':name,**{k:fmt(v) if k!='n_trials' else v
                                                  for k,v in metrics(y,preds[name]).items()}})
pooled_metrics.append({'procedure_id':'C_EXISTING_B2_CONTEXT',
                       **{k:fmt(v) if k!='n_trials' else v
                          for k,v in metrics(y,C_predictions).items()}})

participant_mse={}
participant_sse={}
participant_n=np.asarray([np.sum(people==p) for p in participants],dtype=np.int64)
for name in PROCEDURES:
    sq=(preds[name]-y)**2
    participant_sse[name]=np.asarray([np.sum(sq[people==p],dtype=np.float64)
                                     for p in participants],dtype=np.float64)
    participant_mse[name]=participant_sse[name]/participant_n
balanced_rows=[{'procedure_id':name,
                'participant_balanced_RMSE_W_per_kg':fmt(np.sqrt(np.mean(participant_mse[name])))}
               for name in ('C+D','C+D+V')]

comparisons=(('primary','C+D','C+D+V'),('secondary','D','D+V'))
bootstrap_ledger=[]; contrast_rows=[]
for cname,reference,candidate in comparisons:
    ref_sse=counts@participant_sse[reference]
    can_sse=counts@participant_sse[candidate]
    draw_n=counts@participant_n
    differences=np.sqrt(ref_sse/draw_n)-np.sqrt(can_sse/draw_n)
    observed=rmse(y,preds[reference])-rmse(y,preds[candidate])
    ci=np.percentile(differences,[2.5,97.5])
    contrast_rows.append({'contrast':cname,'reference_procedure':reference,
                          'candidate_procedure':candidate,
                          'point_RMSE_difference_W_per_kg':fmt(observed),
                          'CI_2_5_W_per_kg':fmt(ci[0]),
                          'CI_97_5_W_per_kg':fmt(ci[1]),
                          'bootstrap_draws':10000,
                          'interpretation':'secondary_descriptive' if cname=='secondary' else 'primary_exploratory'})
    bootstrap_ledger += [{'contrast':cname,'draw_index':i+1,
                          'RMSE_difference_W_per_kg':fmt(value)}
                         for i,value in enumerate(differences)]

person_primary=[]
for i,p in enumerate(participants):
    a=float(np.sqrt(participant_mse['C+D'][i]));
    b=float(np.sqrt(participant_mse['C+D+V'][i]));
    d=a-b
    person_primary.append({'subject_id':p,'n_trials':int(participant_n[i]),
                           'C_plus_D_RMSE_W_per_kg':fmt(a),
                           'C_plus_D_plus_V_RMSE_W_per_kg':fmt(b),
                           'difference_reference_minus_candidate_W_per_kg':fmt(d),
                           'direction':('V_favoured' if d>0 else
                                        'no_V_favoured' if d<0 else 'exact_tie')})
sign_counts={label:sum(r['direction']==label for r in person_primary)
             for label in ('V_favoured','no_V_favoured','exact_tie')}
check(sum(sign_counts.values())==21,'Participant contrast sign coverage mismatch')
print('Bootstrap comparisons, all 21 participant contrasts, and balanced sensitivity: PASS')

In [ ]:
# 10 — Independent simple-formula metric cross-check and output repeatability gates
# A separate external independent audit of returned ledgers is still required for final N3 approval.
def independent_rmse(values, estimates):
    return float((sum((float(p)-float(a))**2 for a,p in zip(values,estimates))/len(values))**0.5)

for name in PROCEDURES:
    a=independent_rmse(y,preds[name]); b=rmse(y,preds[name])
    check(abs(a-b)<=METRIC_ATOL,f'Internal metric cross-check failed: {name}')
for contrast in contrast_rows:
    expected=independent_rmse(y,preds[contrast['reference_procedure']])-\
             independent_rmse(y,preds[contrast['candidate_procedure']])
    check(abs(expected-float(contrast['point_RMSE_difference_W_per_kg']))<=METRIC_ATOL,
          f'Contrast point estimate recomputation failed: {contrast["contrast"]}')
check(len(bootstrap_ledger)==20000 and len(person_primary)==21 and
      len(pooled_metrics)==5 and len(fit_checks)==(5*4*6*4)+(5*4),
      'Mandatory metric/provenance record count mismatch')
print('Internal producer checks PASS; N3 remains PENDING EXTERNAL INDEPENDENT AUDIT.')

In [ ]:
# 11 — Save complete evidence ledgers, sha256 manifest, and compact result archive
save_csv(OUT/'candidate_inner_scores.csv',candidate_scores,list(candidate_scores[0].keys()))
save_csv(OUT/'selected_outer_models.csv',selected_models,list(selected_models[0].keys()))
save_csv(OUT/'held_out_predictions.csv',predictions,list(predictions[0].keys()))
fit_columns=list(dict.fromkeys(key for r in fit_checks for key in r))
save_csv(OUT/'fitting_and_N1_repeatability.csv',fit_checks,fit_columns)
save_csv(OUT/'pooled_metrics.csv',pooled_metrics,list(pooled_metrics[0].keys()))
save_csv(OUT/'paired_bootstrap_contrasts.csv',contrast_rows,list(contrast_rows[0].keys()))
save_csv(OUT/'bootstrap_draw_differences.csv',bootstrap_ledger,list(bootstrap_ledger[0].keys()))
save_csv(OUT/'participant_balanced_RMSE.csv',balanced_rows,list(balanced_rows[0].keys()))
save_csv(OUT/'primary_per_participant_contrasts.csv',person_primary,list(person_primary[0].keys()))

# A user-supplied executed preflight notebook digest can be added to the external audit
# record later; its absence from the returned EXT-S0 archive is explicitly documented.
run_info={'status':'PASS_LOCAL_PENDING_INDEPENDENT_N3',
          'experiment':'E3.4', 'approved_method_release':METHOD_ID,
          'source_sha256':SOURCE_SHA,'accepted_EXT_S0_results_zip_sha256':EXT_ZIP_SHA,
          'original_reference_zip_sha256':REFERENCE_ZIP_SHA,
          'fixed_E5_2_population_draws_sha256':BOOTSTRAP_SHA,
          'approved_amendment_sha256':AMENDMENT_SHA,
          'approved_E3_4_annex_sha256':E34_SHA,
          'notebook_file_sha256':sha256_file(NOTEBOOK),
          'notebook_code_cell_sha256':hashlib.sha256(''.join(
              ''.join(c.get('source',[])) for c in json.loads(NOTEBOOK.read_text(encoding='utf-8'))['cells']
              if c['cell_type']=='code').encode('utf-8')).hexdigest(),
          'source_and_canonical_mean_D_equal':'PASS',
          'V_math_and_preoutcome_quality':'PASS',
          'grouped_outer_inner_partitions':'PASS',
          'candidate_scores_count':len(candidate_scores),
          'prediction_rows':len(predictions),'n_trials':len(UIDS),'n_participants':21,
          'bootstrap_draws':10000,
          'original_D_reference_max_abs_difference_W_per_kg':fmt(max_diff_D),
          'new_model_N1_repeatability':'PASS',
          'internal_metric_reconstruction':'PASS',
          'independent_external_N3':'PENDING',
          'E3_4_V1_final_gate':'PENDING_INDEPENDENT_AUDIT',
          'previously_inspected_outer_folds':True,
          'evidence_status':'secondary_exploratory_informed_internal_evaluation',
          'executed_at_UTC':datetime.now(timezone.utc).isoformat(),
          'environment':{'python':sys.version.split()[0],'numpy':np.__version__,
                         'scipy':scipy.__version__,'scikit_learn':sklearn.__version__,
                         'h5py':h5py.__version__,'platform':platform.platform()},
          'participant_primary_sign_counts':sign_counts,
          'verification_tolerances':{'feature_degrees':ATOL_DEG,
                                     'prediction_W_per_kg':PRED_ATOL,
                                     'metric_W_per_kg':METRIC_ATOL}}
files=[p for p in OUT.iterdir() if p.is_file() and p.name!='E3_4_run_summary.json']
run_info['files_sha256']={p.name:sha256_file(p) for p in sorted(files)}
(OUT/'E3_4_run_summary.json').write_text(json.dumps(run_info,indent=2)+'\n',encoding='utf-8')

with zipfile.ZipFile(ARCHIVE,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.iterdir()):
        if p.is_file(): z.write(p,arcname=p.name)
print('E3.4 local production completed with passed internal checks.')
print('Saved:',ARCHIVE)
print('Return only this ZIP for independent N3 verification; do not send raw Baček HDF5.')